# Volatility Forecasting in South Africa 🇿🇦

In this assignment you'll build a model to predict stock volatility for the telecommunications company MTN Group.

In [ ]:
%load_ext autoreload
%autoreload 2

from arch.univariate.base import ARCHModelResult



In [ ]:
# Import your libraries here
import pandas as pd
import requests
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd
from config import settings
import numpy as np
import pandas as pd
from arch import arch_model
from data import SQLRepository
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

## Working with APIs

Create a URL to get all the stock data for MTN Group ("MTNOY") from AlphaVantage in JSON format. Be sure to use the https://learn-api.wqu.edu hostname. And don't worry: your submission won't include your API key!

In [ ]:
ticker = "MTNOY"
output_size = "full"
data_type = "json"

api_key = "91d1c83c9aa1ad69edc331e6519632c6d79b225400e969c7dfcfb0f931cb34e0229f854c3f7e4456e2f957c25e8dfd4e1500e7b224bc0f947f2d4b047804bcef73d3031d9ca7e29dafc9ad1f0909d422730fdf9329ed6f03f91235abfe05912f637301dd2e583c2e23725274aaeef29571d8b1dd29c64999423384cf4ebe8c6f"

url = (
    f"https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
    f"function=TIME_SERIES_DAILY&"
    f"symbol={ticker}&"
    f"outputsize={output_size}&"
    f"datatype={data_type}&"
    f"apikey={api_key}"
)

print("url type:", type(url))
print(url)


Create an HTTP request for the URL you created in the previous task. The grader will evaluate your work by looking at the ticker symbol in the "Meta Data" key-value pair in your response.

In [ ]:
response = requests.get(url)

print("response type:", type(response))

Get status code of your response and assign it to the variable response_code.

In [ ]:
response_code = response.status_code

print("code type:", type(response_code))
response_code

## Test-Driven Development

Create a DataFrame df_mtnoy with all the stock data for MTN. Make sure that the DataFrame has the correct type of index and column names. The grader will evaluate your work by looking at the row in df_mtnoy for 6 December 2021.

In [ ]:
data = response.json()

# Extract the time series data
stock_data = data["Time Series (Daily)"]

# Convert to DataFrame
df_mtnoy = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)

# Format index and column names
df_mtnoy.index = pd.to_datetime(df_mtnoy.index)
df_mtnoy.index.name = "date"
df_mtnoy.columns = [col.split(". ")[1] for col in df_mtnoy.columns]

print("df_mtnoy type:", type(df_mtnoy))
df_mtnoy.head()

Connect to the database whose name is stored in the .env file for this project. Be sure to set the check_same_thread argument to False. Assign the connection to the variable connection. The grader will evaluate your work by looking at the database location assigned to connection.

In [ ]:
# Get the database path from the .env file
db_path = settings.db_name  # This assumes .env has DB_NAME or similar, mapped as `db_name` in `settings`

# Connect to the database
connection = sqlite3.connect(db_path, check_same_thread=False)

# Show the connection object
connection

Insert df_mtnoy into your database. The grader will evaluate your work by looking at the first five rows of the MTNOY table in the database.

In [ ]:
# Insert df_mtnoy into the database as table "MTNOY"
num_inserted = df_mtnoy.to_sql(
    name="MTNOY",
    con=connection,
    if_exists="replace",
    index=True
)

#  Return the expected dictionary
{
    "transaction_successful": True,
    "records_inserted": num_inserted
}


### Read the MTNOY table from your database and assign the output to df_mtnoy_read.

In [ ]:
# Read the MTNOY table from the database
df_mtnoy_read = pd.read_sql("SELECT * FROM MTNOY", con=connection, parse_dates=["date"])

# Set 'date' as index (since it was the index originally)
df_mtnoy_read.set_index("date", inplace=True)

# Output required info
print("df_mtnoy_read type:", type(df_mtnoy_read))
print("df_mtnoy_read shape:", df_mtnoy_read.shape)
df_mtnoy_read.head()


## Predicting Volatility

### Prepare Data

In [ ]:
# Create a Series y_mtnoy with the 2,500 most recent returns for MTN. The grader will evaluate your work by looking at the volatility for 9 August 2022.

# Sort the DataFrame by date (ascending)
df_mtnoy_read.sort_index(ascending=True, inplace=True)

# Create "return" column as percent change of close price
df_mtnoy_read["return"] = df_mtnoy_read["close"].pct_change() * 100

# Extract the last 2,500 non-null returns
y_mtnoy = df_mtnoy_read["return"].dropna().iloc[-2500:]

print("y_mtnoy type:", type(y_mtnoy))
print("y_mtnoy shape:", y_mtnoy.shape)
y_mtnoy.head()

In [ ]:
# Calculate daily volatility for y_mtnoy, and assign the result to mtnoy_daily_volatility.

mtnoy_daily_volatility = y_mtnoy.std()

print("mtnoy_daily_volatility type:", type(mtnoy_daily_volatility))
print("MTN Daily Volatility:", mtnoy_daily_volatility)

In [ ]:
# Calculate the annual volatility for y_mtnoy, and assign the result to mtnoy_annual_volatility.

mtnoy_annual_volatility = mtnoy_daily_volatility * np.sqrt(252)

print("mtnoy_annual_volatility type:", type(mtnoy_annual_volatility))
print("MTN Annual Volatility:", mtnoy_annual_volatility)

In [ ]:
# Create a time series line plot for y_mtnoy. Be sure to label the x-axis "Date", the y-axis "Returns", and use the title "Time Series of MTNOY Returns".

# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Plot `y_mtnoy` on `ax`
y_mtnoy.plot(ax=ax)

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Returns")

# Add title
plt.title("Time Series of MTNOY Returns");

Create an ACF plot of the squared returns for MTN. Be sure to label the x-axis "Lag [days]", the y-axis "Correlation Coefficient", and use the title "ACF of MTNOY Squared Returns".

In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared returns
plot_acf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("ACF of MTNOY Squared Returns");

Create a PACF plot of the squared returns for MTN. Be sure to label the x-axis "Lag [days]", the y-axis "Correlation Coefficient", and use the title "PACF of MTNOY Squared Returns".

In [ ]:
# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create PACF of squared returns
plot_pacf(y_mtnoy**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel("Correlation Coefficient")

# Add title
plt.title("PACF of MTNOY Squared Returns");

In [ ]:
## Create a training set y_mtnoy_train that contains the first 80% of the observations in y_mtnoy.

cutoff_test = int(len(y_mtnoy) * 0.8)
y_mtnoy_train = y_mtnoy.iloc[:cutoff_test]

print("y_mtnoy_train type:", type(y_mtnoy_train))
print("y_mtnoy_train shape:", y_mtnoy_train.shape)
y_mtnoy_train.head()

## Build Model

In [ ]:
## Build and fit a GARCH model using the data in y_mtnoy. Try different values for p and q, using the summary to assess its performance. 
## The grader will evaluate whether your model is the correct data type.

# Build and train model
model = arch_model(
    y_mtnoy_train,
    p=1,
    q=1,
    rescale=False
).fit(disp=0)

print("model type:", type(model))

# Show model summary
model.summary()

In [ ]:
## Plot the standardized residuals for your model. Be sure to label the x-axis "Date", the y-axis "Value", 
## and use the title "MTNOY GARCH Model Standardized Residuals".

fig, ax = plt.subplots(figsize=(15, 6))

# Plot standardized residuals
model.std_resid.plot(ax=ax, label="Standardized Residuals")

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("MTNOY GARCH Model Standardized Residuals");

In [ ]:
## Create an ACF plot of the squared, standardized residuals of your model. Be sure to label the x-axis "Lag [days]", 
## the y-axis "Correlation Coefficient", and use the title "ACF of MTNOY GARCH Model Standardized Residuals".


# Create `fig` and `ax`
fig, ax = plt.subplots(figsize=(15, 6))

# Create ACF of squared, standardized residuals
plot_acf(model.std_resid**2, ax=ax)

# Add axis labels
plt.xlabel("Lag [days]")
plt.ylabel('Correlation Coefficient')

# Add title
plt.title("ACF of MTNOY GARCH Model Standardized Residuals");
